# Wactorz in a notebook

The IMU example run from Jupyter: the system runs on the notebook's own event loop through `wactorz.serve()`, readings are published from a cell, anomalies are read back from a cell, and the running agent is inspected in-process.

Before running:

- a broker on `localhost:1883` (`docker compose up -d mosquitto` from the repository root);
- `pip install wactorz` in the kernel's environment;
- the kernel started with this folder as its working directory, so `agent.py` and `model.py` are importable;
- a free port for the dashboard: Jupyter takes 8888, the dashboard's default, so the notebook uses 8890.

In [1]:
import asyncio
import json
import sys
from pathlib import Path

import aiomqtt

import wactorz

# Make this folder importable even when Jupyter was started elsewhere.
HERE = Path.cwd() if (Path.cwd() / "agent.py").exists() else Path(__file__).parent
sys.path.insert(0, str(HERE))
print("python", sys.version.split()[0], "| wactorz", wactorz.__version__)

python 3.10.11 | wactorz 0.7.0


## 1. Train the model

Stands in for your own training; it writes `imu_model.pkl` beside the agent.

In [2]:
from train import main as train

train()

wrote C:\Users\pkasn\Documents\wactorz_prod\wactorz\examples\imu_anomaly\imu_model.pkl


## 2. The agent, as a function

Before anything connects, the function is callable and testable on its own. `spec.build()` makes the actor around it, which here only supplies `options`, `persist` and `recall`.

In [3]:
from agent import detect

from wactorz.agents.function_agent import spec_of

spec = spec_of(detect)
me = spec.build(persistence_dir=str(HERE / "state"))
print("name:", spec.name, "| subscribes:", spec.subscribes, "| publishes:", spec.publishes)
print("normal ->", detect({"ax": 0.1, "ay": 0.0, "az": 1.0}, me))
print("jolt   ->", detect({"ax": 9.0, "ay": -7.5, "az": 1.0}, me))

name: imu-anomaly | subscribes: ('sensors/imu/#',) | publishes: anomalies/imu
normal -> None
jolt   -> {'score': 39.75, 'reading': {'ax': 9.0, 'ay': -7.5, 'az': 1.0}}


## 3. Start the system on the notebook's loop

`serve()` returns when the system stops, so it runs as a task. `minimal=True` means the monitor, the dashboard and our agent, no model key needed. The host keeps its signals; the task is cancelled to stop.

Jupyter itself listens on port 8888, which is also the dashboard's default, so the dashboard is put on another port here with `monitor_port`.

In [4]:
DASHBOARD_PORT = 8890  # 8888 is Jupyter's own port

system_task = asyncio.create_task(
    wactorz.serve(
        agents=[detect], minimal=True, monitor_port=DASHBOARD_PORT, state_dir=str(HERE / "state")
    )
)
await asyncio.sleep(4)  # let the broker connection, the dashboard and the agent come up
print("running:", not system_task.done(), f"| dashboard: http://localhost:{DASHBOARD_PORT}/")

2026-10-04 13:30:30,160 [INFO] wactorz.app: LLM: openai/gemma4:26b | temperature=0.0
2026-10-04 13:30:30,165 [WARNING] wactorz.app: [startup] API_KEY is 13 characters, under the 16 this expects. Sign-in attempts are throttled, so guessing it through the form is slow — but requests carrying the key as a header are not, so a short one is worth replacing. Something like `openssl rand -hex 32` gives a key nobody has to remember.
2026-10-04 13:30:30,171 [INFO] wactorz.core.mqtt_publisher: [MQTT] Publisher started → localhost:1883 | client_id=wactorz-pub-eb8d0fcac2e3 | outbox_db=C:\Users\pkasn\Documents\wactorz_prod\wactorz\examples\imu_anomaly\state\mqtt_outbox.db
2026-10-04 13:30:30,173 [INFO] wactorz.app: TopicBus initialised
2026-10-04 13:30:30,180 [INFO] wactorz.core.persistence.db: [Persistence] SQLite opened: C:\Users\pkasn\Documents\wactorz_prod\wactorz\examples\imu_anomaly\state\wactorz.db
2026-10-04 13:30:30,182 [INFO] wactorz.core.persistence.db: [Persistence] Schema v1 ready
2026

## 4. Listen for anomalies

A background task collects what the agent publishes on `anomalies/imu`.

In [5]:
anomalies: list[dict] = []


async def collect() -> None:
    async with aiomqtt.Client("localhost", 1883) as client:
        await client.subscribe("anomalies/imu")
        async for message in client.messages:
            anomalies.append(json.loads(message.payload.decode()))


collector = asyncio.create_task(collect())
await asyncio.sleep(1)

## 5. Publish readings

Normal motion around gravity, with a jolt every 25th reading.

In [6]:
import numpy as np

rng = np.random.default_rng()


async def publish_readings(count: int = 100, period: float = 0.05) -> None:
    async with aiomqtt.Client("localhost", 1883) as client:
        for i in range(count):
            if i % 25 == 24:
                reading = {"ax": 9.0, "ay": -7.5, "az": 1.0}
            else:
                ax, ay, az = rng.normal((0.0, 0.0, 1.0), (0.3, 0.3, 0.2))
                reading = {
                    "ax": round(float(ax), 3),
                    "ay": round(float(ay), 3),
                    "az": round(float(az), 3),
                }
            await client.publish("sensors/imu/notebook", json.dumps(reading))
            await asyncio.sleep(period)


await publish_readings()
await asyncio.sleep(1)
print(f"{len(anomalies)} anomalies (expected 4)")
for item in anomalies:
    print(item)

4 anomalies (expected 4)
{'score': 39.75, 'reading': {'ax': 9.0, 'ay': -7.5, 'az': 1.0}}
{'score': 39.75, 'reading': {'ax': 9.0, 'ay': -7.5, 'az': 1.0}}
{'score': 39.75, 'reading': {'ax': 9.0, 'ay': -7.5, 'az': 1.0}}
{'score': 39.75, 'reading': {'ax': 9.0, 'ay': -7.5, 'az': 1.0}}


## 6. Inspect the running agent

The system runs in this process, so the actor itself is reachable: its counters, and what it persisted.

In [7]:
from wactorz.web import runtime

actor = runtime.registry.find_by_name("imu-anomaly")
print("messages processed:", actor.metrics.messages_processed)
print("anomalies persisted:", actor.recall("anomalies_total"))
print("state:", actor.state.value, "| uptime:", round(actor.metrics.uptime, 1), "s")

messages processed: 457
anomalies persisted: 5
state: running | uptime: 35.2 s


## 7. Ask it as a task

What chat does with `@imu-anomaly {...}`: a task message, answered with the function's result.

In [8]:
from wactorz import Actor, Message, MessageType

replies: list[Message] = []


class Inbox(Actor):
    """A real actor, so the reply has a mailbox to land in."""

    async def handle_message(self, msg: Message) -> None:
        replies.append(msg)


inbox = Inbox(name="notebook", persistence_dir=str(HERE / "state"))
await runtime.registry.register(inbox)
await inbox.start()
await runtime.registry.deliver(
    actor.actor_id,
    Message(
        type=MessageType.TASK,
        sender_id=inbox.actor_id,
        reply_to=inbox.actor_id,
        payload={"_task_id": "t1", "ax": 9.0, "ay": -7.5, "az": 1.0},
    ),
)
await asyncio.sleep(0.5)
print(replies[-1].payload if replies else "no reply yet")

2026-10-04 13:31:09,937 [INFO] wactorz.core.actor: [notebook] Actor created with id=4a3171eb-f9f8-53e5-810e-be289887f7c3
2026-10-04 13:31:09,942 [INFO] wactorz.core.registry: [Registry] Registered notebook (4a3171eb)
2026-10-04 13:31:09,944 [INFO] wactorz.core.actor: [notebook] Actor started.
{'score': 39.75, 'reading': {'ax': 9.0, 'ay': -7.5, 'az': 1.0}, '_task_id': 't1'}


## 8. Stop

Cancelling the task runs the same shutdown as Ctrl-C would: agents stop, state is written, the dashboard closes.

In [9]:
collector.cancel()
await inbox.stop()
system_task.cancel()
await asyncio.gather(collector, system_task, return_exceptions=True)
print("stopped:", system_task.done())

2026-10-04 13:31:19,302 [INFO] wactorz.core.actor: [notebook] Actor stopped.


CancelledError: 